In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys

project_root = '/content/drive/MyDrive/ai_enginner/job_search/AI/'
cache_dir = '/content/drive/MyDrive/ai_enginner/job_search/AI/cache/'
os.environ['HF_HOME'] = cache_dir
sys.path.append(project_root)
sys.path.append(cache_dir)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
# 더미 데이터 생성
user_info = {
        "cat_kewd": "84", # 직무 코드
        "keydownAccess": "", # 검색 키워드
        "loc_mcd": "101000", # 지역 코드
        "exp_cd": "1", # 경력 코드
        "exp_none": "y", # 경력 무관
        "edu_none": "y", # 학력 무관
        "edu_min": "6", # 학력 최소
        "edu_max": "9", # 학력 최대
        "search_done": "y", # 검색 완료
    }

# 사람인 채용 정보 html 추출
html_content = crawl_job_html_from_saramin(user_info)

# #content > div.recruit_list_renew > div 영역 내의 채용 정보 추출
job_data = extract_job_major_info(html_content)

# 채용 정보 요약 출력
print_job_summary(job_data)

In [ ]:
print('-'*10, '유사도 검색 시작', '-'*10)
# device 선택
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device: ', device)

---------- 유사도 검색 시작 ----------
device:  cpu


In [ ]:
# 모델 로드
print('모델 로드 시작')
model_name = 'dragonkue/snowflake-arctic-embed-l-v2.0-ko'

# device에 따른 설정
model = SentenceTransformer(model_name).to(device)
print('모델 로드 완료')

모델 로드 시작
모델 로드 완료


In [ ]:
documents = [{'회사명': '', '채용제목': '[시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용', '직무분야': '백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31', '근무지': '서울구로구', '경력': '신입', '학력': '학력무관', '마감일': '~ 09/08(월)', '지원링크': 'https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51266037&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0'}, {'회사명': '', '채용제목': '영어회화 가능자', '직무분야': '백엔드/서버개발,기술지원,데이터엔지니어,IT컨설팅,SE(시스템엔지니어)외수정일 25/08/29', '근무지': '서울양천구', '경력': '신입', '학력': '학력무관', '마감일': '내일마감', '지원링크': 'https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51688737&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0'}]

print('documents를 문자열로 변환 시작')
# documents가 딕셔너리 리스트인 경우 문자열로 변환
if documents and isinstance(documents[0], dict):
    # 딕셔너리를 문자열로 변환 (회사명, 채용제목, 직무분야 등을 조합)
    document_strings = []
    for doc in documents:
        # 딕셔너리의 모든 키-값 쌍을 "키: 값" 형태로 조합
        doc_parts = [f"{key}: {value}" for key, value in doc.items() if value]
        doc_str = " | ".join(doc_parts)
        document_strings.append(doc_str)
    print('document_strings: ', document_strings)
    documents_for_embedding = document_strings
else:
    # 이미 문자열 리스트인 경우
    documents_for_embedding = documents
print('documents를 문자열로 변환 완료')

documents를 문자열로 변환 시작
document_strings:  ['채용제목: [시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용 | 직무분야: 백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31 | 근무지: 서울구로구 | 경력: 신입 | 학력: 학력무관 | 마감일: ~ 09/08(월) | 지원링크: https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51266037&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0', '채용제목: 영어회화 가능자 | 직무분야: 백엔드/서버개발,기술지원,데이터엔지니어,IT컨설팅,SE(시스템엔지니어)외수정일 25/08/29 | 근무지: 서울양천구 | 경력: 신입 | 학력: 학력무관 | 마감일: 내일마감 | 지원링크: https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51688737&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0']
documents를 문자열로 변환 완료


In [ ]:
query = "웹 개발자 신입 채용"

# 임베딩 계산
print('임베딩 계산 시작')
query_embeddings = model.encode(query, prompt_name="query")
document_embeddings = model.encode(documents_for_embedding)
print('임베딩 계산 완료')

임베딩 계산 시작
임베딩 계산 완료


In [ ]:
# 코사인 유사도 점수 계산
print('코사인 유사도 점수 계산 시작')
scores = model.similarity(query_embeddings, document_embeddings)
print('코사인 유사도 점수 계산 완료')

코사인 유사도 점수 계산 시작
코사인 유사도 점수 계산 완료


In [ ]:
documents

[{'회사명': '',
  '채용제목': '[시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용',
  '직무분야': '백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31',
  '근무지': '서울구로구',
  '경력': '신입',
  '학력': '학력무관',
  '마감일': '~ 09/08(월)',
  '지원링크': 'https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51266037&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0'},
 {'회사명': '',
  '채용제목': '영어회화 가능자',
  '직무분야': '백엔드/서버개발,기술지원,데이터엔지니어,IT컨설팅,SE(시스템엔지니어)외수정일 25/08/29',
  '근무지': '서울양천구',
  '경력': '신입',
  '학력': '학력무관',
  '마감일': '내일마감',
  '지원링크': 'https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51688737&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0'}]

In [ ]:
# 문서와 유사도 점수 쌍 생성 및 정렬
print('문서와 유사도 점수 쌍 생성 및 정렬 시작')
doc_score_pairs = list(zip(documents_for_embedding, scores[0]))
doc_score_pairs = sorted(doc_score_pairs, key=lambda x: x[1], reverse=True)
print('문서와 유사도 점수 쌍 생성 및 정렬 완료')

문서와 유사도 점수 쌍 생성 및 정렬 시작
문서와 유사도 점수 쌍 생성 및 정렬 완료


In [ ]:
print(doc_score_pairs[0][0])
print(doc_score_pairs[0][1])

채용제목: [시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용 | 직무분야: 백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31 | 근무지: 서울구로구 | 경력: 신입 | 학력: 학력무관 | 마감일: ~ 09/08(월) | 지원링크: https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51266037&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0
tensor(0.5022)


In [ ]:
print('문서와 유사도 점수 쌍 출력 시작')
for document, score in doc_score_pairs:
    print(f"{score:.4f}: {document[:100]}...")
print('문서와 유사도 점수 쌍 출력 완료')

문서와 유사도 점수 쌍 출력 시작
0.5022: 채용제목: [시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용 | 직무분야: 백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31 | 근무지: 서울구로구 |...
0.3716: 채용제목: 영어회화 가능자 | 직무분야: 백엔드/서버개발,기술지원,데이터엔지니어,IT컨설팅,SE(시스템엔지니어)외수정일 25/08/29 | 근무지: 서울양천구 | 경력: 신입 | ...
문서와 유사도 점수 쌍 출력 완료
